# CryptoML-SBox — Eksperimen Skala Besar XGBoost & AES Modifikasi (Jobdesk Mahdi)

Notebook ini dirancang khusus untuk dijalankan di **Google Colab** atau **Kaggle** agar tidak membebani CPU/GPU dan penyimpanan laptop lokal.

**Yang dilakukan notebook ini:**
1. Mengkloning repo `MahdiHbb1/cryptoml-sbox` dan memasang dependensi ringan.
2. Menjalankan unit test `pytest` untuk verifikasi modul AES-128 & XGBoost.
3. Melatih & melakukan *hyperparameter tuning* **XGBoost** (`--tune`) pada dataset besar (otomatis memakai `data/sbox_dataset.csv` jika sudah tersedia dari Yafi, atau membangkitkan `1.500` sampel beragam), lalu menyaring **15.000 kandidat S-box** untuk memilih `results/sbox_xgb.npy` terbaik.
4. Membandingkan Metrik 1 (**AES Standar vs K44 vs S-Box XGBoost**) serta menguji enkripsi & dekripsi *lossless* citra **Mandrill** (`data/mandril_color.jpg`).
5. Menyimpan hasil (`results/sbox_xgb.npy` dan `results/xgb_metrics.json`) agar siap diunduh / di-commit ke GitHub.

In [ ]:
# Sel 1: Clone repo & install dependensi
import os, subprocess

REPO_URL = "https://github.com/MahdiHbb1/cryptoml-sbox.git"
BRANCH = "feat/aes-and-xgboost"  # Ubah ke 'main' setelah PR di-merge

if not os.path.exists("cryptoml-sbox"):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL], check=True)
else:
    subprocess.run(["git", "-C", "cryptoml-sbox", "pull", "origin", BRANCH], check=True)

%cd cryptoml-sbox
!pip install -q -r requirements.txt

In [ ]:
# Sel 2: Jalankan unit test untuk memastikan seluruh fungsi valid
!python -m pytest -v

In [ ]:
# Sel 3: Training + Tuning XGBoost skala besar & penyaringan 15.000 kandidat S-box
import json
import numpy as np
from src.aes_affine_generator import build_k44_sbox, build_standard_aes_sbox
from src.train_xgboost import evaluate_sbox_default, run_xgboost_pipeline

summary = run_xgboost_pipeline(
    dataset_path="data/sbox_dataset.csv",
    out_sbox_path="results/sbox_xgb.npy",
    out_metrics_path="results/xgb_metrics.json",
    n_train_samples=1500,       # Dipakai bila data/sbox_dataset.csv belum tersedia
    n_screen_candidates=15000,  # Jumlah kandidat baru yang disaring cepat oleh XGBoost
    top_k=40,                   # 40 kandidat teratas dievaluasi penuh Metrik 1
    tune=True,                  # Aktifkan RandomizedSearchCV
    seed=42,
    pool_mode="aes_affine",
)

print("=== PERFORMA MODEL XGBOOST ===")
ml = summary["ml_metrics"]
print(f"R2={ml['r2']:.4f} | MAE={ml['mae']:.4f} | RMSE={ml['rmse']:.4f} | Train Time={ml['train_time_sec']:.2f}s")
print("Best params:", ml["best_params"])

print("\n=== PERBANDINGAN METRIK 1: AES STANDAR vs K44 vs S-BOX XGBOOST ===")
for label, sb in [
    ("AES Standar", build_standard_aes_sbox()),
    ("K44 Alamsyah", build_k44_sbox()),
    ("S-Box XGBoost (Mahdi)", np.load("results/sbox_xgb.npy")),
]:
    m = evaluate_sbox_default(sb)
    print(
        f"{label:24s} | Bij={m['bijective']} | NL={m['nl']} | "
        f"SAC={m['sac_mean']:.5f} (max_err={m['sac_max_err']:.5f}) | "
        f"BIC-NL={m['bic_nl']} | BIC-SAC={m['bic_sac']:.5f} | "
        f"DU={m['du']} | DAP={m['dap']:.6f} | LAP={m['lap']:.4f} | FP={m['fixed_points']}"
    )

In [ ]:
# Sel 4: Uji enkripsi & dekripsi lossless pada citra Mandrill memakai S-Box XGBoost
import matplotlib.pyplot as plt
from PIL import Image
from src.aes_cipher_runner import decrypt_image_array, encrypt_image_array

def shannon_entropy(arr):
    p = np.bincount(arr.ravel(), minlength=256) / arr.size
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

sbox_xgb = np.load("results/sbox_xgb.npy")
plain = np.array(Image.open("data/mandril_color.jpg").convert("RGB"), dtype=np.uint8)
plain_flip = plain.copy()
plain_flip[-1, -1, -1] ^= np.uint8(1)  # Ubah 1 bit di piksel terakhir

key = b"CryptoMLSBox2026"
iv = b"MandrillIV123456"

c1 = encrypt_image_array(plain, sbox=sbox_xgb, key=key, iv=iv, mode="CBC2")
c2 = encrypt_image_array(plain_flip, sbox=sbox_xgb, key=key, iv=iv, mode="CBC2")
rec = decrypt_image_array(c1, sbox=sbox_xgb, key=key, iv=iv, orig_shape=plain.shape, mode="CBC2")

npcr = float(np.mean(c1 != c2) * 100.0)
uaci = float(np.mean(np.abs(c1.astype(np.int32) - c2.astype(np.int32)) / 255.0) * 100.0)

print(f"Lossless decrypt : {np.array_equal(rec, plain)}")
print(f"Entropy Plain    : {shannon_entropy(plain):.5f}")
print(f"Entropy Cipher   : {shannon_entropy(c1):.5f}")
print(f"NPCR (C1 vs C2)  : {npcr:.4f}%")
print(f"UACI (C1 vs C2)  : {uaci:.4f}%")

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, img_data, title in zip(axes, [plain, c1, rec], ["Mandrill Asli", "Terenkripsi (AES-CBC2 + S-Box XGBoost)", "Hasil Dekripsi"]):
    ax.imshow(img_data)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Sel 5: Unduh hasil dengan deteksi aman untuk Kaggle vs Google Colab
import os, shutil, sys
from pathlib import Path

is_kaggle = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle/working").exists()
is_colab = ("COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules) and not is_kaggle

if is_kaggle:
    from IPython.display import FileLink, display
    for fname in ["sbox_xgb.npy", "xgb_metrics.json"]:
        src = Path("results") / fname
        dst = Path("/kaggle/working") / fname
        shutil.copy2(src, dst)
        display(FileLink(f"results/{fname}"))
    print("Selesai! Klik link di atas atau ambil dari panel kanan Kaggle: Output -> /kaggle/working/")
elif is_colab:
    from google.colab import files
    files.download("results/sbox_xgb.npy")
    files.download("results/xgb_metrics.json")
else:
    print("Lingkungan lokal — file tersimpan di 'results/sbox_xgb.npy' dan 'results/xgb_metrics.json'.")